In [ ]:
# 📦 Install required libraries (run only once in Colab)
!pip install opencv-python-headless scikit-image scikit-learn matplotlib

# 📁 Imports
import os
import cv2
import numpy as np
from scipy.ndimage import rotate
from sklearn.cluster import DBSCAN
from skimage import exposure
import matplotlib.pyplot as plt

# 📁 Define input/output folders
input_root = "/content/drive/MyDrive/Datasets/UTFVP/20151211-utwente-vingervein/dataset/data"
output_root = "/content/drive/MyDrive/Datasets/UTFVP_ROI_Hist"
os.makedirs(output_root, exist_ok=True)

# === CONFIG ===
RESIZED_SHAPE = (128, 256)
ROI_SIZE = (60, 128)
PADDING_ROWS = 3
THRESHOLD_VAR = 38
ROTATION_THRESHOLD = 15.0

# === CLAHE ===
def apply_clahe(image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# === Resize ===
def resize_image(img):
    return cv2.resize(img, RESIZED_SHAPE[::-1], interpolation=cv2.INTER_CUBIC)

# === Prewitt Filters ===
def prewitt_mask_upper():
    return np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

def prewitt_mask_lower():
    return -prewitt_mask_upper()

def apply_edge_filter(image, upper_mask, lower_mask):
    h = image.shape[0] // 2
    upper = cv2.filter2D(image[:h, :], -1, upper_mask)
    lower = cv2.filter2D(image[h:, :], -1, lower_mask)
    filtered = np.vstack([upper, lower])
    return cv2.normalize(filtered, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def find_edges(filtered):
    top_edge = np.argmax(filtered, axis=0)
    bottom_edge = filtered.shape[0] - 1 - np.argmax(filtered[::-1], axis=0)
    return top_edge, bottom_edge

def coarse_binarization(img):
    filtered = apply_edge_filter(img, prewitt_mask_upper(), prewitt_mask_lower())
    top_edge, bottom_edge = find_edges(filtered)
    bin_img = np.zeros_like(img, dtype=np.uint8)
    for col in range(img.shape[1]):
        if top_edge[col] < bottom_edge[col]:
            bin_img[top_edge[col]:bottom_edge[col], col] = 255
    return bin_img, top_edge, bottom_edge

def check_abnormal_case(top_edge, bottom_edge, thr=THRESHOLD_VAR):
    n = len(top_edge)
    var_up_I = np.var(top_edge[:n//2])
    var_up_II = np.var(top_edge[n//2:])
    var_low_I = np.var(bottom_edge[:n//2])
    var_low_II = np.var(bottom_edge[n//2:])
    if var_up_I > thr and var_up_II > thr:
        return "upper"
    elif var_low_I > thr and var_low_II > thr:
        return "lower"
    return None

def elaborate_binarization(img):
    bin_img, top_edge, bottom_edge = coarse_binarization(img)
    case = check_abnormal_case(top_edge, bottom_edge)
    if case == "upper":
        padded = np.pad(img, ((PADDING_ROWS, 0), (0, 0)), mode='constant')
    elif case == "lower":
        padded = np.pad(img, ((0, PADDING_ROWS), (0, 0)), mode='constant')
    else:
        return bin_img, top_edge, bottom_edge, img
    re_bin_img, top_edge, bottom_edge = coarse_binarization(padded)
    if case == "upper":
        re_bin_img = re_bin_img[PADDING_ROWS:, :]
    elif case == "lower":
        re_bin_img = re_bin_img[:-PADDING_ROWS, :]
    return re_bin_img, top_edge, bottom_edge, padded

def clean_middle_line(middle_line):
    X = np.arange(len(middle_line)).reshape(-1, 1)
    Y = middle_line.reshape(-1, 1)
    points = np.hstack([X, Y])
    clustering = DBSCAN(eps=10, min_samples=10).fit(points)
    core_mask = clustering.labels_ != -1
    return middle_line[core_mask], X[core_mask].flatten()

def orientation_angle_filtered(x, y):
    if len(x) < 2:
        return 0
    x_mean, y_mean = np.mean(x), np.mean(y)
    k = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
    return np.degrees(np.arctan(k))

def rotate_image(image, angle):
    return rotate(image, -angle, reshape=False, order=3, mode='constant', cval=0)

def roi_extraction(image, top_edge, bottom_edge):
    h, w = image.shape
    middle_line = (top_edge + bottom_edge) // 2
    y_center = int(np.median(middle_line))
    y1 = max(0, y_center - ROI_SIZE[0] // 2)
    y2 = y1 + ROI_SIZE[0]
    start = w // 4
    end = 3 * w // 4
    ref_block = image[:, start:end]
    projection = np.sum(ref_block, axis=0)
    c = start + np.argmax(projection)
    x1 = max(0, c - ROI_SIZE[1] // 2)
    x2 = x1 + ROI_SIZE[1]
    roi = image[max(0, y1):min(y2, h), max(0, x1):min(x2, w)]
    roi_resized = cv2.resize(roi, ROI_SIZE[::-1], interpolation=cv2.INTER_CUBIC)
    roi_equalized = exposure.equalize_hist(roi_resized)
    return (roi_equalized * 255).astype(np.uint8)

def process_finger_vein_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError("Failed to load image.")
    enhanced = apply_clahe(img)
    resized = resize_image(enhanced)
    bin_img, top_edge, bottom_edge, padded = elaborate_binarization(resized)
    middle_line = (top_edge + bottom_edge) // 2
    clean_y, clean_x = clean_middle_line(middle_line)
    angle = orientation_angle_filtered(clean_x, clean_y)
    corrected = rotate_image(padded, angle) if abs(angle) > ROTATION_THRESHOLD else padded
    roi = roi_extraction(corrected, top_edge, bottom_edge)
    return roi

# === Batch ROI Extraction ===
for subject_id in sorted(os.listdir(input_root)):
    subject_folder = os.path.join(input_root, subject_id)
    if not os.path.isdir(subject_folder):
        continue
    output_subject_folder = os.path.join(output_root, subject_id)
    os.makedirs(output_subject_folder, exist_ok=True)
    for filename in sorted(os.listdir(subject_folder)):
        if filename.lower().endswith(".png"):
            input_img_path = os.path.join(subject_folder, filename)
            short_name = "_".join(filename.split("_")[:3]) + ".png"
            output_img_path = os.path.join(output_subject_folder, short_name)
            try:
                roi = process_finger_vein_image(input_img_path)
                cv2.imwrite(output_img_path, roi)
                print(f"✅ Saved ROI: {output_img_path}")
            except Exception as e:
                print(f"❌ Failed to process {filename} → {e}")
